In [1]:
# Imports 
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

In [8]:
# Load
SEED = 123
df = pd.read_csv("../Dati/Processed/dataset_processed_quantile1_sentences.csv")
X, y = df[["text_bert"]], df["binary_label"].values
groups = df["topic_id"].values  # keep triplets together across splits

In [9]:
# Pipeline + nested grid search
# char n-grams only on text; vectorizer refit per fold -> no leakage
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

feats = ColumnTransformer([
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                             min_df=0.025, sublinear_tf=True), "text_bert"),
])
pipe = Pipeline([
    ("feats", feats),
    ("clf", LinearSVC(class_weight="balanced", random_state=SEED, max_iter=8000)),
])
grid = {"clf__C": [0.3, 0.5, 1.0]}  # no C=2.0: avoid low-regularization overfit

# Group-aware CV, recomputed at each level -> valid on subsets
outer = StratifiedGroupKFold(n_splits=5)
inner = StratifiedGroupKFold(n_splits=5)

gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=inner, n_jobs=-1)
nested = cross_val_score(gs, X, y, groups=groups, scoring="f1_macro",
                         cv=outer, params={"groups": groups})
print(f"char-only nested f1_macro = {nested.mean():.4f} ± {nested.std():.4f}")

char-only nested f1_macro = 0.6549 ± 0.0197


In [10]:
# Best config + OOF predictions (same group-aware folds)
gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=outer, n_jobs=-1).fit(X, y, groups=groups)
print(gs.best_params_)
oof = cross_val_predict(gs.best_estimator_, X, y, cv=outer, groups=groups)

{'clf__C': 1.0}


In [11]:
# Classification report + confusion matrix 
print(classification_report(y, oof, digits=3))
print(confusion_matrix(y, oof))

              precision    recall  f1-score   support

           0      0.589     0.476     0.527       208
           1      0.761     0.834     0.796       416

    accuracy                          0.715       624
   macro avg      0.675     0.655     0.661       624
weighted avg      0.704     0.715     0.706       624

[[ 99 109]
 [ 69 347]]
